<a href="https://colab.research.google.com/github/syedhasannadeem/test.project/blob/main/Rag_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -qU langchain-pinecone langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 9.3 MB/s eta 0:00:00


In [3]:
from google.colab import userdata

from pinecone import Pinecone, ServerlessSpec

pinecone_api_key = userdata.get('PINECONE_API_KEY')

pc = Pinecone(api_key=pinecone_api_key)

In [26]:
import time

index_name = "onine-rag-project"  # existing index name

# Use the existing index
print(f"Index '{index_name}' already exists. Using the existing index.")
index = pc.Index(index_name)


Index 'onine-rag-project' already exists. Using the existing index.


In [49]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [50]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [51]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocalate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [52]:
len(documents)

10

In [54]:
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['a8f28a42-b60b-4ed4-a93b-5e37670c51f6',
 '5280f148-0b1d-4ced-ab48-f4e8389f734e',
 '329899de-e584-4858-935a-5d690ac1baca',
 'a1190613-9477-420b-b98d-aac2c2cd008e',
 'e3724d08-0f99-4129-841e-344699bed8de',
 'f8234566-b111-47df-a120-ab481b1a0a7f',
 'e234f9b7-bbd2-42ac-a47a-2b0d8efb5eee',
 '25a555ef-cc44-4d9f-9ce4-09ecde9a03fe',
 '36500411-4e97-4c55-846b-2b5a82c60a20',
 'b6857116-6f99-4d99-aa71-d1fcecfb7dc0']

In [55]:
results = vector_store.similarity_search(
    "how much down stock market",
    k=2,
    filter={"source": "news"},

    )
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* The stock market is down 500 points today due to fears of a recession. [{'source': 'news'}]
* The stock market is down 500 points today due to fears of a recession. [{'source': 'news'}]


In [100]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [104]:
def answer_to_user(query: str):
    try:
        # Perform similarity search
        vector_results = vector_store.similarity_search(query, k=2)

        # Check if any results were found
        if not vector_results:
            return "I'm sorry, but I couldn't find any relevant references to answer your query."

        # Prepare input for the language model (LLM)
        llm_input = f"""
        Answer the following user query: "{query}".
        Here are some references to use in your answer: {vector_results}.
        """

        # Generate answer using the LLM
        final_answer = llm.invoke(llm_input)
        return final_answer

    except AttributeError as e:
        return f"An AttributeError occurred: {str(e)}. Please check if 'vector_store' and 'llm' are properly initialized."
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

# Example query
query = "how much down stock market?"

# Call the function and print the answer
answer = answer_to_user(query)
print(answer.content)



Based on the provided news articles, the stock market is down 500 points today.  The reason cited is fear of a recession.

